In [ ]:
import pandas as pd
# load civic dataset (path explicit to workspace)
df = pd.read_csv("Day-6/civic_issue_dataset.csv")
df.head()

,Report_ID,Date_Submitted,Hour_of_Day,Issue_Type,Department,Zone,Severity_Score,Severity_Label,Num_Upvotes,Has_Photo,Has_Voice_Note,Population_Density,Dept_Current_Workload,Weather_Severity_Index,Priority_Score,Resolution_Time_Hours,Citizen_Satisfaction_Score
0,RPT-01001,2023-01-01,3,Broken Sidewalk,Public Works,Zone E,3,High,9,0,1,772,32,1,19.11,51.6,4.6
1,RPT-01002,2023-01-01,17,Drainage,Public Works,Zone E,4,Critical,17,0,0,1532,52,5,27.67,42.6,2.8
2,RPT-01003,2023-01-02,6,Graffiti,Maintenance,Zone A,1,Low,1,0,1,4953,42,1,19.21,62.1,1.0
3,RPT-01004,2023-01-03,12,Broken Sidewalk,Public Works,Zone C,3,High,7,1,0,4202,35,3,25.50,43.1,6.3
4,RPT-01005,2023-01-03,14,Trash/Waste,Sanitation,Zone B,1,Low,3,1,0,3334,18,3,17.04,30.7,6.7


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

target = 'Severity_Label'
feature_cols = [c for c in df.columns if c not in ['Report_ID', 'Date_Submitted', target]]

X = df[feature_cols].copy()
y = df[target].astype(str).copy()

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
if 'Hour_of_Day' in categorical_cols:
    categorical_cols.remove('Hour_of_Day')
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

x_train, x_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=22, stratify=y
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=32,
    n_jobs=-1
)

model.fit(x_train, y_train)
accuracy = model.score(x_test, y_test)
print('Test accuracy:', accuracy)

Test accuracy: 0.98


In [4]:
def predict_severity(issue_type, department, zone, has_photo=0, has_voice_note=0,
                     hour_of_day=12, num_upvotes=0, population_density=1000,
                     dept_current_workload=30, weather_severity_index=1):
    sample = pd.DataFrame([{
        'Issue_Type': issue_type,
        'Department': department,
        'Zone': zone,
        'Has_Photo': has_photo,
        'Has_Voice_Note': has_voice_note,
        'Hour_of_Day': hour_of_day,
        'Num_Upvotes': num_upvotes,
        'Population_Density': population_density,
        'Dept_Current_Workload': dept_current_workload,
        'Weather_Severity_Index': weather_severity_index
    }])

    sample_encoded = pd.get_dummies(sample, columns=[c for c in sample.select_dtypes(include=['object']).columns.tolist()], drop_first=True)
    sample_encoded = sample_encoded.reindex(columns=X_encoded.columns, fill_value=0)
    return model.predict(sample_encoded)[0]

prediction = predict_severity('Drainage', 'Public Works', 'Zone E', has_photo=0, has_voice_note=0, hour_of_day=17, num_upvotes=5, population_density=1532, dept_current_workload=52, weather_severity_index=5)
prediction

'Low'